In [1]:
import numpy as np
import pandas as pd

In [ ]:
import pandas as pd
import re

file_path = "./normal_run_data.txt"
pattern = re.compile(
    r"Timestamp:\s+([\d\.]+)\s+ID:\s+([0-9a-fA-F]+)\s+\d+\s+DLC:\s+(\d+)\s+(.+)"
)

rows = []
with open(file_path, "r") as f:
    for line in f:
        match = pattern.search(line)
        if match:
            timestamp = float(match.group(1))
            can_id = match.group(2)
            dlc = int(match.group(3))
            data_bytes = match.group(4).strip().split()

            data_bytes = data_bytes[:dlc] + ["00"] * (8 - dlc)
            rows.append([timestamp, can_id, dlc] + data_bytes)

col_names = ["Timestamp", "ID", "DLC"] + [f"Byte{i}" for i in range(1, 9)]
df = pd.DataFrame(rows, columns=col_names)

df["ID"] = df["ID"].apply(lambda x: int(x, 16))
for i in range(1, 9):
    df[f"Byte{i}"] = df[f"Byte{i}"].apply(lambda x: int(x, 16))

output_path = "./normal_run_data.csv"
df.to_csv(output_path, index=False)

df.head()

,Timestamp,ID,DLC,Byte1,Byte2,Byte3,Byte4,Byte5,Byte6,Byte7,Byte8
0,1.479121e+09,848,8,5,40,132,102,109,0,0,162
1,1.479121e+09,704,8,20,0,0,0,0,0,0,0
2,1.479121e+09,1072,8,0,0,0,0,0,0,0,0
3,1.479121e+09,1201,8,0,0,0,0,0,0,0,0
4,1.479121e+09,497,8,0,0,0,0,0,0,0,0


In [ ]:
import pandas as pd

col_names = [
    "Timestamp", "ID", "DLC",
    "Byte1", "Byte2", "Byte3", "Byte4",
    "Byte5", "Byte6", "Byte7", "Byte8", "Flag"
]

Attack_List = ["Normal", "DoS", "Fuzzy", "Gear", "RPM"]

Normal = pd.read_csv("./normal_run_data.csv", low_memory=False)
DoS = pd.read_csv("./DoS_dataset.csv", low_memory=False, header=None, names=col_names)
Fuzzy = pd.read_csv("./Fuzzy_dataset.csv", low_memory=False, header=None, names=col_names)
Gear = pd.read_csv("./gear_dataset.csv", low_memory=False, header=None, names=col_names)
RPM = pd.read_csv("./RPM_dataset.csv", low_memory=False, header=None, names=col_names)

In [ ]:
import numpy as np
import pandas as pd

BYTE_COLS = [f"Byte{i}" for i in range(1, 9)]

def fast_clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    dlc = pd.to_numeric(df["DLC"], errors="coerce").fillna(0).astype(np.int16).clip(0, 8)
    df["DLC"] = dlc

    bytes_str = df[BYTE_COLS].astype(str).apply(lambda s: s.str.strip())

    bytes_upper = bytes_str.apply(lambda s: s.str.upper())
    mask_tr = bytes_upper.isin(["T", "R"])               
    arr  = bytes_upper.to_numpy(copy=False)
    mask = mask_tr.to_numpy(copy=False)
    any_tr   = mask.any(axis=1)
    first_ix = mask.argmax(axis=1)                      
    flag_from_bytes = np.where(any_tr, arr[np.arange(arr.shape[0]), first_ix], np.nan)
    flag_from_bytes = pd.Series(flag_from_bytes, index=df.index)

    flag_series = df.get("Flag", pd.Series(index=df.index, dtype=object)).astype(str).str.strip().str.upper()
    df["Flag"] = flag_series.where(flag_series.isin(["T", "R"]), flag_from_bytes).fillna("R")

    bytes_str = bytes_str.mask(mask_tr)

    pos = np.arange(1, 9, dtype=np.int8)
    over_mask = pos > dlc.to_numpy()[:, None]              # shape: (n, 8)
    bytes_str = bytes_str.where(~over_mask, other="00")

    bytes_str = bytes_str.fillna("00").replace({"": "00", "nan": "00", "None": "00"})
    bytes_norm = bytes_str.apply(lambda s: s.str.lower().str.zfill(2))

    hex_map = {f"{i:02x}": i for i in range(256)}
    bytes_int = bytes_norm.apply(lambda s: s.map(hex_map)).fillna(0).astype(np.uint8)
    df[BYTE_COLS] = bytes_int

    df["ID"] = df["ID"].astype(str).str.strip().apply(lambda s: int(s, 16)).astype("Int64")

    df["Timestamp"] = pd.to_numeric(df["Timestamp"], errors="coerce")
    print("Done")
    return df

DoS   = fast_clean(DoS)
Fuzzy = fast_clean(Fuzzy)
Gear  = fast_clean(Gear)
RPM   = fast_clean(RPM)

In [ ]:
DoS.to_csv("./DoS_dataset_clean.csv", index=False)
Fuzzy.to_csv("./Fuzzy_dataset_clean.csv", index=False)
Gear.to_csv("./gear_dataset_clean.csv", index=False)
RPM.to_csv("./RPM_dataset_clean.csv", index=False)